# RETFound Final Run — Ablations + External Validation + Full Plots

Loads **RETFound Baseline** / **Enhanced RETFound** checkpoints, trains ablations A1–A3, runs external validation, and exports:

- Train/val **loss** and **QWK** curves
- Confusion matrices
- Referable-DR **ROC** curves
- **McNemar** tests (Baseline vs Enhanced)

### Attach on Kaggle
1. APTOS 2019  
2. RETFound CFP weights  
3. Saved training outputs (`M0_best.pt`, `M2_best.pt`, `splits.json`, histories if available)  
4. External dataset (Messidor-2 / DDR)  

**GPU required.** Edit paths in Config, then Run All → **Save Version**.

In [ ]:
# ========================= CONFIG =========================
from pathlib import Path

class CFG:
    IS_KAGGLE = Path("/kaggle/input").exists()

    DATA_DIR = Path("/kaggle/input/aptos2019-blindness-detection") if IS_KAGGLE else Path("./data/aptos2019")
    TRAIN_CSV = DATA_DIR / "train.csv"
    TRAIN_IMG_DIR = DATA_DIR / "train_images"

    # EDIT these
    SAVED_OUT = Path("/kaggle/input/retfound-dr-outputs/outputs") if IS_KAGGLE else Path("./outputs")
    RETFOUND_WEIGHTS = Path("/kaggle/input/retfound-cfp-weights/RETFound_cfp_weights.pth") if IS_KAGGLE else Path("./weights/RETFound_cfp_weights.pth")
    EXTERNAL_DIR = Path("/kaggle/input/messidor2") if IS_KAGGLE else Path("./data/messidor2")
    EXTERNAL_CSV = None
    EXTERNAL_NAME = "Messidor-2"

    OUT_DIR = Path("/kaggle/working/outputs") if IS_KAGGLE else Path("./outputs_final")
    FIG_DIR = OUT_DIR / "figures"

    NUM_CLASSES = 5
    IMG_SIZE = 224
    BATCH_SIZE = 16
    NUM_WORKERS = 2
    SEED = 42

    RUN_INTERNAL_RESCORE = True
    RUN_ABLATIONS = True
    RUN_EXTERNAL = True
    # Compute train QWK each epoch (slower but better thesis curves)
    COMPUTE_TRAIN_QWK = True

    ABLATION_EPOCHS = 15
    LR_M2 = 2e-4
    WEIGHT_DECAY = 0.05
    WARMUP_EPOCHS = 2
    FOCAL_GAMMA = 2.0
    LORA_R, LORA_ALPHA, LORA_DROPOUT = 8, 16, 0.05
    LORA_TARGETS = ["qkv"]
    MS_BLOCKS = (7, 15, 23)

print("SAVED_OUT:", CFG.SAVED_OUT)
print("EXTERNAL_DIR:", CFG.EXTERNAL_DIR)
print("RUN_ABLATIONS:", CFG.RUN_ABLATIONS, "| RUN_EXTERNAL:", CFG.RUN_EXTERNAL)

In [ ]:
# ========================= INSTALLS =========================
import sys, subprocess, math, json, random, shutil

def pip_install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

pip_install(["timm", "scikit-learn", "opencv-python-headless"])

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import timm
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix, roc_auc_score,
    classification_report, cohen_kappa_score, roc_curve, auc
)
from scipy.stats import chi2
import matplotlib.pyplot as plt

CFG.OUT_DIR.mkdir(parents=True, exist_ok=True)
CFG.FIG_DIR.mkdir(parents=True, exist_ok=True)

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(CFG.SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))

In [ ]:
# ========================= RESOLVE PATHS =========================
def resolve_weights():
    cands = [CFG.RETFOUND_WEIGHTS,
             Path("/kaggle/input/retfound-cfp-weights/RETFound_cfp_weights.pth"),
             Path("/kaggle/input/retfound/RETFound_cfp_weights.pth")]
    for p in cands:
        if p.is_file():
            return str(p)
    if CFG.IS_KAGGLE:
        hits = list(Path("/kaggle/input").rglob("*RETFound*cfp*.pth"))
        hits += list(Path("/kaggle/input").rglob("*retfound*.pth"))
        if hits:
            return str(hits[0])
    raise FileNotFoundError("RETFound weights not found")


def resolve_saved_out():
    cands = [CFG.SAVED_OUT]
    if CFG.IS_KAGGLE:
        cands += list(Path("/kaggle/input").glob("*"))
        cands += list(Path("/kaggle/input").glob("*/outputs"))
    for root in cands:
        root = Path(root)
        if (root / "M0_best.pt").is_file() and (root / "splits.json").is_file():
            return root
        if (root / "outputs" / "M0_best.pt").is_file():
            return root / "outputs"
    raise FileNotFoundError(f"Could not find M0_best.pt + splits.json under {CFG.SAVED_OUT}")


WEIGHTS_PATH = resolve_weights()
SAVED_OUT = resolve_saved_out()
print("WEIGHTS_PATH:", WEIGHTS_PATH)
print("SAVED_OUT:", SAVED_OUT)

for name in ["M0_best.pt", "M2_best.pt", "splits.json",
             "M0_history.csv", "M2_history.csv", "comparison_test.csv", "stats.json"]:
    src = SAVED_OUT / name
    if src.exists():
        shutil.copy(src, CFG.OUT_DIR / name)
        print("copied", name)

In [ ]:
# ========================= APTOS + SPLITS =========================
df = pd.read_csv(CFG.TRAIN_CSV).rename(columns={"id_code": "image_id", "diagnosis": "label"})
df["image_id"] = df["image_id"].astype(str)
df["label"] = df["label"].astype(int)

def find_image_path(image_id, root=CFG.TRAIN_IMG_DIR):
    for ext in (".png", ".jpg", ".jpeg", ".PNG", ".JPG"):
        p = Path(root) / f"{image_id}{ext}"
        if p.is_file():
            return str(p)
    return None

df["path"] = df["image_id"].map(find_image_path)
df = df.dropna(subset=["path"]).reset_index(drop=True)

splits = json.loads((CFG.OUT_DIR / "splits.json").read_text())
train_df = df[df["image_id"].isin(splits["train"])].reset_index(drop=True)
val_df   = df[df["image_id"].isin(splits["val"])].reset_index(drop=True)
test_df  = df[df["image_id"].isin(splits["test"])].reset_index(drop=True)
print("split sizes:", len(train_df), len(val_df), len(test_df))

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

class FundusDataset(Dataset):
    def __init__(self, frame, train=False, img_size=224):
        self.df, self.train, self.img_size = frame, train, img_size
    def __len__(self): return len(self.df)
    def _augment(self, img):
        if random.random() < 0.5: img = img.transpose(Image.FLIP_LEFT_RIGHT)
        if random.random() < 0.5: img = img.rotate(random.uniform(-15, 15), resample=Image.BILINEAR)
        if random.random() < 0.5:
            from PIL import ImageEnhance
            img = ImageEnhance.Brightness(img).enhance(random.uniform(0.9, 1.1))
            img = ImageEnhance.Contrast(img).enhance(random.uniform(0.9, 1.1))
        return img
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["path"]).convert("RGB").resize((self.img_size, self.img_size), Image.BICUBIC)
        if self.train: img = self._augment(img)
        arr = np.asarray(img).astype(np.float32) / 255.0
        arr = (arr - np.array(IMAGENET_MEAN)) / np.array(IMAGENET_STD)
        return torch.from_numpy(arr).permute(2, 0, 1).float(), int(row["label"])

def make_loaders(tr, va, te, bs=CFG.BATCH_SIZE):
    return (
        DataLoader(FundusDataset(tr, True), batch_size=bs, shuffle=True, num_workers=CFG.NUM_WORKERS, pin_memory=True),
        DataLoader(FundusDataset(va, False), batch_size=bs, shuffle=False, num_workers=CFG.NUM_WORKERS, pin_memory=True),
        DataLoader(FundusDataset(te, False), batch_size=bs, shuffle=False, num_workers=CFG.NUM_WORKERS, pin_memory=True),
    )

train_loader, val_loader, test_loader = make_loaders(train_df, val_df, test_df)

def class_weights_from_df(frame, n_classes=5, device="cpu"):
    counts = frame["label"].value_counts().reindex(range(n_classes), fill_value=0).values.astype(np.float32)
    counts = np.maximum(counts, 1.0)
    return torch.tensor(counts.sum() / (n_classes * counts), dtype=torch.float32, device=device)

CW = class_weights_from_df(train_df, CFG.NUM_CLASSES, DEVICE)

In [ ]:
# ========================= METRICS + PLOTTING HELPERS =========================
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None):
        super().__init__(); self.gamma, self.weight = gamma, weight
    def forward(self, logits, target):
        ce = F.cross_entropy(logits, target, weight=self.weight, reduction="none")
        pt = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()

class CoralOrdinalLoss(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__(); self.num_classes = num_classes
    def forward(self, ordinal_logits, y):
        levels = torch.arange(self.num_classes - 1, device=y.device).expand(y.size(0), -1)
        return F.binary_cross_entropy_with_logits(ordinal_logits, (y.unsqueeze(1) > levels).float())

def compute_metrics(y_true, y_pred, y_prob=None):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    out = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "qwk": float(cohen_kappa_score(y_true, y_pred, weights="quadratic")),
    }
    yt = (y_true >= 2).astype(int); yp = (y_pred >= 2).astype(int)
    out["referable_acc"] = float(accuracy_score(yt, yp))
    if y_prob is not None:
        try:
            out["referable_auroc"] = float(roc_auc_score(yt, y_prob[:, 2:].sum(axis=1)))
        except ValueError:
            out["referable_auroc"] = float("nan")
    out["confusion_matrix"] = confusion_matrix(y_true, y_pred, labels=list(range(CFG.NUM_CLASSES))).tolist()
    return out

def cosine_lr(optimizer, epoch, total_epochs, base_lr, warmup=1):
    if epoch < warmup:
        lr = base_lr * (epoch + 1) / max(1, warmup)
    else:
        t = (epoch - warmup) / max(1, total_epochs - warmup)
        lr = 0.5 * base_lr * (1 + math.cos(math.pi * t))
    for g in optimizer.param_groups: g["lr"] = lr
    return lr

def plot_cm(y_true, y_pred, title, save_path):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(CFG.NUM_CLASSES)))
    fig, ax = plt.subplots(figsize=(5, 4.5))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title(title); ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_xticks(range(CFG.NUM_CLASSES)); ax.set_yticks(range(CFG.NUM_CLASSES))
    for i in range(CFG.NUM_CLASSES):
        for j in range(CFG.NUM_CLASSES):
            ax.text(j, i, int(cm[i, j]), ha="center", va="center", fontsize=9)
    fig.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout(); fig.savefig(save_path, dpi=160); plt.show()
    return cm

def plot_train_val_curves(history_df, title_prefix, save_prefix):
    """Plot loss and QWK curves from a history dataframe."""
    h = history_df.copy()
    if "epoch" not in h.columns:
        h["epoch"] = np.arange(len(h))

    # Loss
    fig, ax = plt.subplots(figsize=(7, 4))
    if "train_loss" in h: ax.plot(h["epoch"], h["train_loss"], marker="o", label="Train loss")
    if "val_loss" in h: ax.plot(h["epoch"], h["val_loss"], marker="s", label="Val loss")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.set_title(f"{title_prefix} — Loss")
    ax.grid(True, alpha=0.3); ax.legend(); plt.tight_layout()
    fig.savefig(CFG.FIG_DIR / f"{save_prefix}_loss.png", dpi=160); plt.show()

    # QWK
    fig, ax = plt.subplots(figsize=(7, 4))
    if "train_qwk" in h: ax.plot(h["epoch"], h["train_qwk"], marker="o", label="Train QWK")
    if "val_qwk" in h: ax.plot(h["epoch"], h["val_qwk"], marker="s", label="Val QWK")
    # fallback for older history files (only val qwk/acc/f1)
    if "val_qwk" not in h and "qwk" in h: ax.plot(h["epoch"], h["qwk"], marker="s", label="Val QWK")
    ax.set_xlabel("Epoch"); ax.set_ylabel("QWK"); ax.set_title(f"{title_prefix} — QWK")
    ax.grid(True, alpha=0.3); ax.legend(); plt.tight_layout()
    fig.savefig(CFG.FIG_DIR / f"{save_prefix}_qwk.png", dpi=160); plt.show()

def plot_referable_roc(curves, title, save_path):
    """curves: list of dicts with keys name, y_true, y_score"""
    fig, ax = plt.subplots(figsize=(6, 5))
    for c in curves:
        yt = (np.asarray(c["y_true"]) >= 2).astype(int)
        score = np.asarray(c["y_score"])
        try:
            fpr, tpr, _ = roc_curve(yt, score)
            roc_auc = auc(fpr, tpr)
            ax.plot(fpr, tpr, lw=2, label=f"{c['name']} (AUC={roc_auc:.3f})")
        except ValueError:
            print("ROC skipped for", c["name"], "(only one class present)")
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
    ax.set_title(title); ax.legend(loc="lower right"); ax.grid(True, alpha=0.3)
    plt.tight_layout(); fig.savefig(save_path, dpi=160); plt.show()

def plot_multiclass_ovr_roc(y_true, y_prob, title, save_path, class_names=None):
    y_true = np.asarray(y_true); y_prob = np.asarray(y_prob)
    class_names = class_names or [f"Grade {i}" for i in range(CFG.NUM_CLASSES)]
    fig, ax = plt.subplots(figsize=(6.5, 5))
    for i in range(CFG.NUM_CLASSES):
        yt = (y_true == i).astype(int)
        if yt.min() == yt.max():
            continue
        fpr, tpr, _ = roc_curve(yt, y_prob[:, i])
        ax.plot(fpr, tpr, lw=1.8, label=f"{class_names[i]} (AUC={auc(fpr, tpr):.3f})")
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set_xlabel("FPR"); ax.set_ylabel("TPR"); ax.set_title(title)
    ax.legend(fontsize=8, loc="lower right"); ax.grid(True, alpha=0.3)
    plt.tight_layout(); fig.savefig(save_path, dpi=160); plt.show()

def mcnemar_test(y_true, pred_a, pred_b, mode="referable"):
    y_true = np.asarray(y_true); pred_a = np.asarray(pred_a); pred_b = np.asarray(pred_b)
    if mode == "referable":
        t = (y_true >= 2).astype(int)
        a = (pred_a >= 2).astype(int)
        b = (pred_b >= 2).astype(int)
    else:  # exact grade match
        t = y_true; a = pred_a; b = pred_b
        a_correct = a == t; b_correct = b == t
        b01 = int(np.sum(~a_correct & b_correct))
        b10 = int(np.sum(a_correct & ~b_correct))
        stat = (abs(b01 - b10) - 1) ** 2 / max(b01 + b10, 1)
        return {"mode": mode, "b01_B_fixes": b01, "b10_B_breaks": b10,
                "stat": float(stat), "p_value": float(chi2.sf(stat, 1))}
    a_correct = a == t; b_correct = b == t
    b01 = int(np.sum(~a_correct & b_correct))  # A wrong, B right
    b10 = int(np.sum(a_correct & ~b_correct))  # A right, B wrong
    stat = (abs(b01 - b10) - 1) ** 2 / max(b01 + b10, 1)
    return {"mode": mode, "b01_Enhanced_fixes": b01, "b10_Enhanced_breaks": b10,
            "stat": float(stat), "p_value": float(chi2.sf(stat, 1))}

def bootstrap_delta_qwk(y_true, pred_a, pred_b, n_boot=2000, seed=42):
    rng = np.random.default_rng(seed); n = len(y_true); deltas = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        qa = cohen_kappa_score(y_true[idx], pred_a[idx], weights="quadratic")
        qb = cohen_kappa_score(y_true[idx], pred_b[idx], weights="quadratic")
        deltas.append(qb - qa)
    deltas = np.asarray(deltas)
    lo, hi = np.percentile(deltas, [2.5, 97.5])
    return {"delta_qwk": float(np.mean([cohen_kappa_score(y_true, pred_b, weights="quadratic") -
                                        cohen_kappa_score(y_true, pred_a, weights="quadratic")])),
            "ci95_low": float(lo), "ci95_high": float(hi),
            "ci_excludes_0": bool(lo > 0 or hi < 0)}

print("Plot/stats helpers ready.")

In [ ]:
# ========================= MODELS =========================
def load_vit_backbone(num_classes=0, weights_path=""):
    model = timm.create_model("vit_large_patch16_224", pretrained=(weights_path == ""),
                             num_classes=num_classes, global_pool="token")
    if weights_path:
        try: ckpt = torch.load(weights_path, map_location="cpu", weights_only=False)
        except TypeError: ckpt = torch.load(weights_path, map_location="cpu")
        state = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt
        cleaned, model_sd = {}, model.state_dict()
        for k, v in state.items():
            nk = k.replace("module.", "")
            if nk.startswith("head") or nk.startswith("fc_norm"): continue
            if nk in model_sd and model_sd[nk].shape == v.shape: cleaned[nk] = v
        model.load_state_dict(cleaned, strict=False)
        print(f"Loaded RETFound weights | matched={len(cleaned)}")
    return model

class M0RetFound(nn.Module):
    def __init__(self, num_classes=5, weights_path=""):
        super().__init__()
        self.backbone = load_vit_backbone(num_classes, weights_path)
    def forward(self, x): return self.backbone(x)

class MultiScaleFusionHead(nn.Module):
    def __init__(self, dim=1024, num_classes=5, n_scales=3):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(dim * n_scales, dim), nn.GELU(), nn.Dropout(0.1))
        self.classifier = nn.Linear(dim, num_classes)
        self.ordinal = nn.Linear(dim, num_classes - 1)
        self.referable = nn.Linear(dim, 1)
    def forward(self, feats):
        h = self.proj(torch.cat(feats, dim=-1))
        return {"logits": self.classifier(h), "ordinal": self.ordinal(h),
                "referable": self.referable(h).squeeze(-1)}

class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r=8, alpha=16, dropout=0.05):
        super().__init__()
        self.base = base
        for p in self.base.parameters(): p.requires_grad = False
        self.scaling = alpha / r
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.lora_A = nn.Linear(base.in_features, r, bias=False)
        self.lora_B = nn.Linear(r, base.out_features, bias=False)
        nn.init.kaiming_uniform_(self.lora_A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B.weight)
    def forward(self, x):
        return self.base(x) + self.lora_B(self.lora_A(self.dropout(x))) * self.scaling

def inject_lora_timm_vit(model, r=8, alpha=16, dropout=0.05, target_names=("qkv",)):
    n = 0
    for name, module in list(model.named_modules()):
        if name.split(".")[-1] not in target_names or not isinstance(module, nn.Linear): continue
        parent_path = name.rsplit(".", 1)
        parent = model if len(parent_path) == 1 else model.get_submodule(parent_path[0])
        child = name if len(parent_path) == 1 else parent_path[1]
        setattr(parent, child, LoRALinear(module, r=r, alpha=alpha, dropout=dropout)); n += 1
    return n

class M2RetFound(nn.Module):
    def __init__(self, num_classes=5, weights_path="", ms_blocks=(7, 15, 23),
                 lora_r=8, lora_alpha=16, lora_dropout=0.05, lora_targets=None):
        super().__init__()
        self.ms_blocks = tuple(ms_blocks)
        self.backbone = load_vit_backbone(0, weights_path)
        for p in self.backbone.parameters(): p.requires_grad = False
        inject_lora_timm_vit(self.backbone, r=lora_r, alpha=lora_alpha, dropout=lora_dropout,
                             target_names=tuple(lora_targets or ["qkv"]))
        dim = getattr(self.backbone, "embed_dim", 1024)
        self.head = MultiScaleFusionHead(dim, num_classes, len(self.ms_blocks))
        self._hooks, self._cache = [], {}
        self._register_hooks()
    def _register_hooks(self):
        def make_hook(i):
            def hook(_m, _i, out):
                self._cache[i] = 0.5 * (out[:, 0] + out[:, 1:].mean(dim=1))
            return hook
        for i in self.ms_blocks:
            self._hooks.append(self.backbone.blocks[i].register_forward_hook(make_hook(i)))
    def forward(self, x):
        self._cache = {}; _ = self.backbone.forward_features(x)
        return self.head([self._cache[i] for i in self.ms_blocks])

class M2Ablation(nn.Module):
    def __init__(self, use_multiscale=True, num_classes=5, weights_path="",
                 ms_blocks=(7, 15, 23), lora_r=8, lora_alpha=16, lora_dropout=0.05):
        super().__init__()
        self.ms_blocks = tuple(ms_blocks) if use_multiscale else (23,)
        self.backbone = load_vit_backbone(0, weights_path)
        for p in self.backbone.parameters(): p.requires_grad = False
        inject_lora_timm_vit(self.backbone, r=lora_r, alpha=lora_alpha, dropout=lora_dropout)
        dim = getattr(self.backbone, "embed_dim", 1024)
        self.head = MultiScaleFusionHead(dim, num_classes, len(self.ms_blocks))
        self._hooks, self._cache = [], {}
        self._register_hooks()
    def _register_hooks(self):
        def make_hook(i):
            def hook(_m, _i, out):
                self._cache[i] = 0.5 * (out[:, 0] + out[:, 1:].mean(dim=1))
            return hook
        for i in self.ms_blocks:
            self._hooks.append(self.backbone.blocks[i].register_forward_hook(make_hook(i)))
    def forward(self, x):
        self._cache = {}; _ = self.backbone.forward_features(x)
        return self.head([self._cache[i] for i in self.ms_blocks])

print("Models ready.")

In [ ]:
# ========================= LOAD / EVAL =========================
@torch.no_grad()
def predict_model(model, loader, kind="baseline"):
    model.eval(); ys, preds, probs = [], [], []
    for x, y in loader:
        x = x.to(DEVICE)
        out = model(x)
        logits = out if kind == "baseline" else out["logits"]
        pr = torch.softmax(logits, dim=-1)
        ys.append(y.numpy()); preds.append(logits.argmax(-1).cpu().numpy()); probs.append(pr.cpu().numpy())
    return np.concatenate(ys), np.concatenate(preds), np.concatenate(probs)

@torch.no_grad()
def evaluate(model, loader, kind="baseline", criterion=None):
    model.eval(); ys, preds, probs, losses = [], [], [], []
    for x, y in loader:
        x, yb = x.to(DEVICE), y.to(DEVICE)
        out = model(x)
        logits = out if kind == "baseline" else out["logits"]
        if criterion is not None:
            losses.append(criterion(logits, yb).item())
        pr = torch.softmax(logits, dim=-1)
        ys.append(y.numpy()); preds.append(logits.argmax(-1).cpu().numpy()); probs.append(pr.cpu().numpy())
    yt, yp, ypr = np.concatenate(ys), np.concatenate(preds), np.concatenate(probs)
    metrics = compute_metrics(yt, yp, ypr)
    if losses: metrics["loss"] = float(np.mean(losses))
    return metrics, yt, yp, ypr

def load_baseline(path=CFG.OUT_DIR / "M0_best.pt"):
    model = M0RetFound(CFG.NUM_CLASSES, WEIGHTS_PATH).to(DEVICE)
    ckpt = torch.load(path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt["model"], strict=False); model.eval(); return model

def load_enhanced(path=CFG.OUT_DIR / "M2_best.pt"):
    model = M2RetFound(CFG.NUM_CLASSES, WEIGHTS_PATH, CFG.MS_BLOCKS,
                       CFG.LORA_R, CFG.LORA_ALPHA, CFG.LORA_DROPOUT, CFG.LORA_TARGETS).to(DEVICE)
    ckpt = torch.load(path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt["model"], strict=False); model.eval(); return model

baseline_model = load_baseline()
enhanced_model = load_enhanced()
print("Loaded Baseline + Enhanced checkpoints.")

# Plot saved main-model histories if present
for tag, fname, title in [
    ("baseline", "M0_history.csv", "RETFound Baseline"),
    ("enhanced", "M2_history.csv", "Enhanced RETFound"),
]:
    p = CFG.OUT_DIR / fname
    if p.exists():
        h = pd.read_csv(p)
        # normalize column names from earlier training script
        rename = {}
        if "qwk" in h.columns and "val_qwk" not in h.columns: rename["qwk"] = "val_qwk"
        if "train_loss" not in h.columns and "loss" in h.columns: rename["loss"] = "train_loss"
        h = h.rename(columns=rename)
        plot_train_val_curves(h, title, f"main_{tag}")
    else:
        print("No history CSV for", title)

## A) APTOS internal re-score + CM + ROC + McNemar

In [ ]:
summary_rows, stats_all = [], {}
ce_eval = nn.CrossEntropyLoss(weight=CW)

if CFG.RUN_INTERNAL_RESCORE:
    m0, yt0, yp0, ypr0 = evaluate(baseline_model, test_loader, "baseline", criterion=ce_eval)
    m2, yt2, yp2, ypr2 = evaluate(enhanced_model, test_loader, "enhanced", criterion=ce_eval)

    internal = pd.DataFrame([
        {"model": "RETFound Baseline", "set": "APTOS-test", **{k: m0[k] for k in ("accuracy", "macro_f1", "qwk", "referable_acc", "referable_auroc")}},
        {"model": "Enhanced RETFound", "set": "APTOS-test", **{k: m2[k] for k in ("accuracy", "macro_f1", "qwk", "referable_acc", "referable_auroc")}},
    ])
    print("===== APTOS INTERNAL TEST ====="); display(internal)
    internal.to_csv(CFG.OUT_DIR / "comparison_internal_rescore.csv", index=False)
    summary_rows.extend(internal.to_dict("records"))

    print("\nBaseline per-class:\n", classification_report(yt0, yp0, digits=4))
    print("Enhanced per-class:\n", classification_report(yt2, yp2, digits=4))

    plot_cm(yt0, yp0, "RETFound Baseline — APTOS test", CFG.FIG_DIR / "cm_baseline_aptos.png")
    plot_cm(yt2, yp2, "Enhanced RETFound — APTOS test", CFG.FIG_DIR / "cm_enhanced_aptos.png")

    # Referable ROC (Baseline vs Enhanced)
    plot_referable_roc([
        {"name": "RETFound Baseline", "y_true": yt0, "y_score": ypr0[:, 2:].sum(1)},
        {"name": "Enhanced RETFound", "y_true": yt2, "y_score": ypr2[:, 2:].sum(1)},
    ], "Referable DR ROC — APTOS test", CFG.FIG_DIR / "roc_referable_aptos.png")

    plot_multiclass_ovr_roc(yt0, ypr0, "Baseline one-vs-rest ROC — APTOS", CFG.FIG_DIR / "roc_ovr_baseline_aptos.png")
    plot_multiclass_ovr_roc(yt2, ypr2, "Enhanced one-vs-rest ROC — APTOS", CFG.FIG_DIR / "roc_ovr_enhanced_aptos.png")

    mc_ref = mcnemar_test(yt0, yp0, yp2, mode="referable")
    mc_exact = mcnemar_test(yt0, yp0, yp2, mode="exact")
    boot = bootstrap_delta_qwk(yt0, yp0, yp2)
    stats_all["aptos"] = {"mcnemar_referable": mc_ref, "mcnemar_exact": mc_exact, "bootstrap_qwk": boot,
                          "baseline_qwk": m0["qwk"], "enhanced_qwk": m2["qwk"]}
    print("McNemar referable:", mc_ref)
    print("McNemar exact-grade:", mc_exact)
    print("Bootstrap ΔQWK:", boot)
else:
    print("Internal rescore skipped")

## B) Ablations A1 / A2 / A3 (with train/val loss + QWK curves)

| ID | Variant |
|----|---------|
| A1 | LoRA + late features + focal |
| A2 | LoRA + multi-scale + focal |
| A3 | LoRA + multi-scale + focal + ordinal + referable |

In [ ]:
def train_ablation(name, use_multiscale=True, use_ordinal=True, use_referable=True, epochs=None):
    epochs = epochs or CFG.ABLATION_EPOCHS
    model = M2Ablation(use_multiscale, CFG.NUM_CLASSES, WEIGHTS_PATH, CFG.MS_BLOCKS,
                       CFG.LORA_R, CFG.LORA_ALPHA, CFG.LORA_DROPOUT).to(DEVICE)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                            lr=CFG.LR_M2, weight_decay=CFG.WEIGHT_DECAY)
    focal = FocalLoss(CFG.FOCAL_GAMMA, CW)
    coral = CoralOrdinalLoss(CFG.NUM_CLASSES)
    bce = nn.BCEWithLogitsLoss()
    ce_mon = nn.CrossEntropyLoss(weight=CW)  # for comparable val/train loss curves

    best = {"qwk": -1.0, "path": str(CFG.OUT_DIR / f"{name}_best.pt")}
    history = []

    for epoch in range(epochs):
        model.train()
        lr = cosine_lr(opt, epoch, epochs, CFG.LR_M2, CFG.WARMUP_EPOCHS)
        losses = []
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            out = model(x)
            loss = focal(out["logits"], y)
            if use_ordinal: loss = loss + 0.5 * coral(out["ordinal"], y)
            if use_referable: loss = loss + 0.3 * bce(out["referable"], (y >= 2).float())
            loss.backward(); opt.step(); losses.append(loss.item())

        # Validation metrics + CE loss for curve
        val_m, *_ = evaluate(model, val_loader, "enhanced", criterion=ce_mon)
        row = {
            "epoch": epoch, "lr": lr,
            "train_loss": float(np.mean(losses)),
            "val_loss": float(val_m.get("loss", np.nan)),
            "val_qwk": float(val_m["qwk"]),
            "val_acc": float(val_m["accuracy"]),
            "val_macro_f1": float(val_m["macro_f1"]),
        }
        if CFG.COMPUTE_TRAIN_QWK:
            tr_m, *_ = evaluate(model, train_loader, "enhanced", criterion=ce_mon)
            row["train_qwk"] = float(tr_m["qwk"])
            # optional: also store CE train loss separately from optim loss
            row["train_ce_loss"] = float(tr_m.get("loss", np.nan))

        history.append(row)
        msg = f"[{name}][{epoch}] train_loss={row['train_loss']:.4f} val_loss={row['val_loss']:.4f} val_qwk={row['val_qwk']:.4f}"
        if "train_qwk" in row: msg += f" train_qwk={row['train_qwk']:.4f}"
        print(msg)

        if val_m["qwk"] > best["qwk"]:
            best["qwk"] = val_m["qwk"]
            torch.save({"model": model.state_dict(), "val": val_m, "epoch": epoch}, best["path"])

    hist_df = pd.DataFrame(history)
    hist_df.to_csv(CFG.OUT_DIR / f"{name}_history.csv", index=False)
    plot_train_val_curves(hist_df, name, f"ablation_{name}")

    ckpt = torch.load(best["path"], map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt["model"], strict=False)
    test_m, yt, yp, ypr = evaluate(model, test_loader, "enhanced")
    print(f"[{name}] TEST", {k: test_m[k] for k in ("accuracy", "macro_f1", "qwk", "referable_auroc")})
    plot_cm(yt, yp, f"{name} — APTOS test", CFG.FIG_DIR / f"cm_{name}.png")
    plot_referable_roc([{"name": name, "y_true": yt, "y_score": ypr[:, 2:].sum(1)}],
                       f"Referable ROC — {name}", CFG.FIG_DIR / f"roc_referable_{name}.png")
    return test_m, yt, yp, ypr, hist_df

ablation_rows, abl_preds = [], {}

if CFG.RUN_ABLATIONS:
    if CFG.RUN_INTERNAL_RESCORE:
        ablation_rows += [
            {"variant": "RETFound Baseline", **{k: m0[k] for k in ("accuracy", "macro_f1", "qwk", "referable_auroc")}},
            {"variant": "Enhanced RETFound (loaded)", **{k: m2[k] for k in ("accuracy", "macro_f1", "qwk", "referable_auroc")}},
        ]
    configs = [
        ("A1_lora_late_focal", dict(use_multiscale=False, use_ordinal=False, use_referable=False)),
        ("A2_lora_ms_focal", dict(use_multiscale=True, use_ordinal=False, use_referable=False)),
        ("A3_lora_ms_ord_ref", dict(use_multiscale=True, use_ordinal=True, use_referable=True)),
    ]
    for name, kw in configs:
        print("\n=====", name, "=====")
        tm, yt, yp, ypr, _ = train_ablation(name, **kw)
        ablation_rows.append({"variant": name, **{k: tm[k] for k in ("accuracy", "macro_f1", "qwk", "referable_auroc")}})
        abl_preds[name] = (yt, yp, ypr)

    abl_df = pd.DataFrame(ablation_rows)
    print("\n===== ABLATION TABLE ====="); display(abl_df)
    abl_df.to_csv(CFG.OUT_DIR / "ablation_results.csv", index=False)

    # Overlay referable ROC for all ablations + mains if available
    roc_list = []
    if CFG.RUN_INTERNAL_RESCORE:
        roc_list += [
            {"name": "Baseline", "y_true": yt0, "y_score": ypr0[:, 2:].sum(1)},
            {"name": "Enhanced", "y_true": yt2, "y_score": ypr2[:, 2:].sum(1)},
        ]
    for name, (yt, yp, ypr) in abl_preds.items():
        roc_list.append({"name": name, "y_true": yt, "y_score": ypr[:, 2:].sum(1)})
    if roc_list:
        plot_referable_roc(roc_list, "Referable DR ROC — APTOS (all models)",
                           CFG.FIG_DIR / "roc_referable_aptos_all.png")
else:
    print("Ablations skipped")

## C) External validation + CM + ROC + McNemar

In [ ]:
def find_ext_image(image_id, roots):
    image_id = str(image_id); stem = Path(image_id).stem
    for root in roots:
        root = Path(root)
        for cand in [image_id, stem]:
            if (root / cand).is_file(): return str(root / cand)
            for ext in (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".PNG", ".JPG"):
                p = root / f"{cand}{ext}"
                if p.is_file(): return str(p)
    return None

def load_external_df(external_dir, csv_path=None):
    external_dir = Path(external_dir)
    assert external_dir.exists(), f"Missing {external_dir}"
    if csv_path is None:
        cands = sorted(external_dir.rglob("*.csv"))
        assert cands, "No CSV — set CFG.EXTERNAL_CSV"
        csv_path = cands[0]
    print("External CSV:", csv_path)
    edf = pd.read_csv(csv_path)
    colmap = {}
    for c in edf.columns:
        cl = c.lower().strip()
        if cl in ("id_code", "image_id", "image", "file", "filename", "id", "img_id"): colmap[c] = "image_id"
        if cl in ("diagnosis", "label", "level", "dr", "grade", "adjudicated_dr_grade", "dr_grade", "retinopathy_grade"): colmap[c] = "label"
    edf = edf.rename(columns=colmap)
    assert {"image_id", "label"} <= set(edf.columns), edf.columns.tolist()
    edf["image_id"] = edf["image_id"].astype(str).str.replace(r"\.(png|jpg|jpeg|tif|tiff)$", "", regex=True)
    edf["label"] = pd.to_numeric(edf["label"], errors="coerce")
    edf = edf.dropna(subset=["label"]); edf["label"] = edf["label"].astype(int)
    edf = edf[edf["label"].between(0, 4)].reset_index(drop=True)
    roots = [external_dir] + [p for p in external_dir.rglob("*") if p.is_dir()]
    edf["path"] = edf["image_id"].apply(lambda i: find_ext_image(i, roots))
    print("External rows:", len(edf), "missing:", int(edf["path"].isna().sum()))
    print("Labels:", edf["label"].value_counts().sort_index().to_dict())
    edf = edf.dropna(subset=["path"]).reset_index(drop=True)
    assert len(edf) > 0
    return edf

if CFG.RUN_EXTERNAL:
    ext_df = load_external_df(CFG.EXTERNAL_DIR, CFG.EXTERNAL_CSV)
    ext_loader = DataLoader(FundusDataset(ext_df, False), batch_size=CFG.BATCH_SIZE,
                            shuffle=False, num_workers=CFG.NUM_WORKERS, pin_memory=True)

    e0, yt_e0, yp_e0, ypr_e0 = evaluate(baseline_model, ext_loader, "baseline")
    e2, yt_e2, yp_e2, ypr_e2 = evaluate(enhanced_model, ext_loader, "enhanced")

    external = pd.DataFrame([
        {"model": "RETFound Baseline", "set": CFG.EXTERNAL_NAME, **{k: e0[k] for k in ("accuracy", "macro_f1", "qwk", "referable_acc", "referable_auroc")}},
        {"model": "Enhanced RETFound", "set": CFG.EXTERNAL_NAME, **{k: e2[k] for k in ("accuracy", "macro_f1", "qwk", "referable_acc", "referable_auroc")}},
    ])
    print("===== EXTERNAL TEST ====="); display(external)
    external.to_csv(CFG.OUT_DIR / "comparison_external.csv", index=False)
    summary_rows.extend(external.to_dict("records"))

    print("\nBaseline per-class:\n", classification_report(yt_e0, yp_e0, digits=4))
    print("Enhanced per-class:\n", classification_report(yt_e2, yp_e2, digits=4))

    plot_cm(yt_e0, yp_e0, f"Baseline — {CFG.EXTERNAL_NAME}", CFG.FIG_DIR / "cm_baseline_external.png")
    plot_cm(yt_e2, yp_e2, f"Enhanced — {CFG.EXTERNAL_NAME}", CFG.FIG_DIR / "cm_enhanced_external.png")

    plot_referable_roc([
        {"name": "RETFound Baseline", "y_true": yt_e0, "y_score": ypr_e0[:, 2:].sum(1)},
        {"name": "Enhanced RETFound", "y_true": yt_e2, "y_score": ypr_e2[:, 2:].sum(1)},
    ], f"Referable DR ROC — {CFG.EXTERNAL_NAME}", CFG.FIG_DIR / "roc_referable_external.png")

    plot_multiclass_ovr_roc(yt_e0, ypr_e0, f"Baseline OvR ROC — {CFG.EXTERNAL_NAME}", CFG.FIG_DIR / "roc_ovr_baseline_external.png")
    plot_multiclass_ovr_roc(yt_e2, ypr_e2, f"Enhanced OvR ROC — {CFG.EXTERNAL_NAME}", CFG.FIG_DIR / "roc_ovr_enhanced_external.png")

    mc_ref_e = mcnemar_test(yt_e0, yp_e0, yp_e2, mode="referable")
    mc_exact_e = mcnemar_test(yt_e0, yp_e0, yp_e2, mode="exact")
    boot_e = bootstrap_delta_qwk(yt_e0, yp_e0, yp_e2)
    stats_all["external"] = {"mcnemar_referable": mc_ref_e, "mcnemar_exact": mc_exact_e,
                             "bootstrap_qwk": boot_e, "baseline_qwk": e0["qwk"], "enhanced_qwk": e2["qwk"]}
    print("External McNemar referable:", mc_ref_e)
    print("External McNemar exact:", mc_exact_e)
    print("External Bootstrap ΔQWK:", boot_e)
else:
    print("External skipped")

## D) Export summary + stats JSON

In [ ]:
if summary_rows:
    summary_df = pd.DataFrame(summary_rows)
    print("===== COMBINED SUMMARY ====="); display(summary_df)
    summary_df.to_csv(CFG.OUT_DIR / "final_summary.csv", index=False)

if Path(CFG.OUT_DIR / "ablation_results.csv").exists():
    print("\nAblations:"); display(pd.read_csv(CFG.OUT_DIR / "ablation_results.csv"))

(CFG.OUT_DIR / "stats_final.json").write_text(json.dumps(stats_all, indent=2))
print("Saved stats_final.json")

manifest = {
    "weights": WEIGHTS_PATH,
    "saved_out": str(SAVED_OUT),
    "external": str(CFG.EXTERNAL_DIR),
    "figures": sorted([p.name for p in CFG.FIG_DIR.glob("*.png")]),
    "outputs": sorted([p.name for p in CFG.OUT_DIR.glob("*") if p.is_file()]),
}
(CFG.OUT_DIR / "final_run_manifest.json").write_text(json.dumps(manifest, indent=2))
print("\nFigures in:", CFG.FIG_DIR)
for p in sorted(CFG.FIG_DIR.glob("*.png")):
    print(" -", p.name)
print("\n*** Save Version on Kaggle now ***")